In [67]:
from ugot import ugot  # UGOT robot SDK
from IPython.display import clear_output  # Jupyter display helpers
import time  # For sleep/timing between motor commands

# Create a robot instance and connect over the network.
got = ugot.UGOT()
got.initialize("10.146.246.43")  # <-- Check if this matches your robot's IP address

# Pre-load vision models onto the UGOT
# All models listed here will be available for the rest of the session.
got.load_models(
    ["color_recognition", "word_recognition", "line_recognition", "apriltag_qrcode"]
)

# Open UGOT camera for later image recognition
got.open_camera()

def line_follow(mult=0.35, speed=20):
    """Follow the detected line by turning proportionally to the line offset."""
    # Read line-tracking information from the robot.
    # `offset` tells how far the detected line is from the center.
    # `type` describes the line/intersection pattern detected.
    # `x` and `y` are the detected line position coordinates.
    offset, line_type, x, y = got.get_single_track_total_info()

    # Convert the line offset into a turning speed.
    # Larger offset -> stronger rotation to re-center on the line.
    rotation_speed = int(offset * mult)

    # Move forward while rotating to stay aligned with the line.
    got.mecanum_move_xyz(x_speed=0, y_speed=speed, z_speed=rotation_speed)

    # Return the detection info so the main program can make decisions.
    return line_type, x, y

try:
    while True:
        got.set_track_recognition_line(line_type=0)
        line_type, x, y = line_follow(mult=0.45, speed=20)

        if line_type in [0, 2]: # 0 = no line detected, 2 = y-intersection detected
            break

    got.mecanum_stop()
    print("Done")

except KeyboardInterrupt:
    got.mecanum_stop()
    print("Done")

10.146.246.43:50051
Done
